# Factoid Standard DPO on Reviewed HEA Data — Qwen2.5 (0.5B or 3B)

Train an ordinary pairwise DPO adapter using the same reviewed HEA-DPO candidate classes and matched experimental protocol. This notebook follows the setup → data → model → frozen reference → training → evaluation → export workflow of `factoid_cal_dpo_qwen25_05b_full_single_answer.ipynb`.

**Run with the `bioasq` Python environment on a CUDA GPU, from this project's notebooks directory.** Run cells from top to bottom. Edit the configuration cell before starting; each run has its own directory. Set `SMOKE_TEST=True` for a small end-to-end check.

- User-reviewed pool: 218 questions / 1,412 candidates after excluding all flagged cases.
- `HEA_DATASET_VARIANT = "reviewed_c3"` uses the original 147 question groups / 860 capped candidates.
- The default `gold_alias_c3` view keeps the same 147 questions and every C2/C1/C0 candidate, but retains only C3 answers matching a recorded BioASQ gold-alias surface. It contains 598 candidates: C3=190, C2=139, C1=219, C0=50. No candidate is added or relabeled.
- Select `MODEL_SIZE = "05b"` or `"3b"` in Run settings. Initialization and fixed reference use that size's strict-extractive SFT `adapter_best_evidence_mrr`: 0.5B step 200 (recorded training-selection MRR 0.45), or 3B step 50 (0.48125). The notebook measures a fresh step-zero baseline with its historical evaluation protocol.
- Objectives: standard pairwise DPO on sampled members of the reviewed class comparisons, or HEA-DPO with hierarchical class preference, C3 equivalence, and canonical realization.
- Evaluation: historical full-dev protocol with multiline prompts, NF4/FP16 loading, one greedy output, no generation cache, and prepared development gold. Reports canonical NLL, MRR/top-1, exact canonical match, and single-answer format validity.

The original annotations and prior trained adapter are preserved. The user has already reviewed and accepted this filtered dataset; no additional approval flag is needed. Independent double-annotation agreement is not claimed.


## 1. Environment and project paths


In [ ]:
from __future__ import annotations
import os
import sys
import gc
import re
import math
import json
import random
import time
import hashlib
import tempfile
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
from datetime import datetime, timezone

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "hea-dpo-matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

def find_project_root(start):
    for parent in [start, *start.parents]:
        for candidate in [parent, parent / "Task-Structured Counterfactual Preference Mining"]:
            if (candidate / "cse_dpo").is_dir() and (candidate / "src").is_dir():
                return candidate.resolve()
    raise RuntimeError("Open this notebook inside the BioASQ_MRES workspace.")

PROJECT_ROOT = find_project_root(Path.cwd())
PILOT_ROOT = PROJECT_ROOT.parent / "HEA-DPO pilot"
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from cse_dpo.generated_bioasq_eval import load_gold_examples
from src.utility.bioasq_format import parse_prediction_items
from src.utility.data import clean_text, list_record_resources
from cse_dpo.hea_historical_evaluation import prepare_historical_dev, evaluate_historical_adapter
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.executable, "| device:", DEVICE)
print("Project:", PROJECT_ROOT)


## 2. Run settings

The selected SFT checkpoint supplies the strict-extraction training instruction. Development uses the historical renderer and preserves instruction line breaks. The reviewed candidate classes, questions, and snippets stay fixed. The original reviewed prompt is retained in each active training record. These candidate labels were reviewed for semantic correctness; they are not newly certified as literal spans under the stricter instruction. This changes initialization and instruction relative to the previous run, so compare with the new step-zero baseline. `METHOD` supports a matched `standard_dpo` baseline and the three HEA component ablations. Standard DPO samples one candidate uniformly from each class in the same recorded class comparison, so question exposure, comparison probabilities, prompts, reference scores, and optimizer budget remain matched. All methods use the same frozen pool and sampler. Both model sizes use BF16 LoRA training without additional quantization, with gradient checkpointing and one candidate activation graph at a time. The 3B run requires more GPU memory and time. During development evaluation, both the training policy and a separate NF4 evaluation model are resident; use `SMOKE_TEST=True` first to check your GPU. `BATCH_QUESTIONS` controls gradient accumulation over sequential questions, not a simultaneous sequence batch. This notebook does not use the SFT trainer's 4-bit training setup.

The reviewed candidate bank is reused for both sizes; these are existing candidates, not newly sampled 3B outputs. Output paths and reference caches are isolated by model size. To switch size, restart the kernel and rerun all cells.

The `gold_alias_c3` match applies Unicode NFKC, whitespace collapse, stripping, and case folding; punctuation and answer tokens must otherwise match the recorded alias. This preserves capitalization variants such as `Prolactinoma` for gold `prolactinoma`, while excluding longer, shorter, and paraphrased C3 answers. Only 39 strict-view questions have multiple C3 members, so the HEA equivalence term is nonzero on fewer questions than in `reviewed_c3`. The output directory and saved configuration include the dataset label.

Full runs evaluate all 160 development questions before training and after each epoch. This development set has canonical/gold answers but no reviewed HEA class annotations, so its loss is **canonical NLL**, not a fabricated validation HEA loss. Your current eight-epoch setting is retained. The final adapter uses the fixed epoch budget; development-selected checkpoints are saved separately. Smoke metrics are explicitly limited to the smoke subset.


In [ ]:
MODEL_SIZE = "05b"  # "05b" or "3b"; restart the kernel after changing this.
METHOD = "standard_dpo"  # This notebook defaults to the matched standard-DPO baseline.
RUN_NAME = None  # None gives a unique UTC timestamp; otherwise use a new label.
SMOKE_TEST = False
SMOKE_TRAIN_QUESTIONS = 4
SMOKE_DEV_QUESTIONS = 2

HEA_DATASET_VARIANT = "gold_alias_c3"  # "gold_alias_c3" or "reviewed_c3"
HEA_DATASETS = {
    "gold_alias_c3": ("reviewed_training_gold_alias_c3_v1/hea_train.jsonl", "gold_alias_c3"),
    "reviewed_c3": ("reviewed_training_v1/hea_train.jsonl", "reviewed_c3"),
}
if HEA_DATASET_VARIANT not in HEA_DATASETS:
    raise ValueError(f"Unknown HEA_DATASET_VARIANT: {HEA_DATASET_VARIANT}")
_train_relative, DATASET_LABEL = HEA_DATASETS[HEA_DATASET_VARIANT]
TRAIN_JSONL = PILOT_ROOT / _train_relative
MODEL_PRESETS = {
    "05b": {"base_name": "Qwen2.5-0.5B-Instruct"},
    "3b": {"base_name": "Qwen2.5-3B-Instruct"},
}
if MODEL_SIZE not in MODEL_PRESETS:
    raise ValueError(f"Unknown MODEL_SIZE: {MODEL_SIZE}; choose 05b or 3b.")
SFT_ROOT = PROJECT_ROOT / f"Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_{MODEL_SIZE}_lora_dropout_005_strict_extractive"
MODEL_REF = SFT_ROOT / "adapter_best_evidence_mrr"
MODEL_LABEL = "strict_extractive_best_mrr"
SFT_PROMPT_INPUT = SFT_ROOT / "trainer_prepared/train_prepared.json"
SFT_SELECTION_SUMMARY = SFT_ROOT / "evidence_mrr_selection/best_summary.json"
BASE_MODEL_REF = PROJECT_ROOT / "models" / MODEL_PRESETS[MODEL_SIZE]["base_name"]
# Intentionally share the same full-dev questions/resources/gold for both sizes.
DEV_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json"
GOLD_INPUT = DEV_INPUT  # Historical evaluator scores the prepared development gold_output.
EVAL_PROMPT_REGISTRY = PROJECT_ROOT / "prompts/factoid_single_answer_aligned.json"
EVAL_PROMPT_ID = "factoid-single-answer-extractive-v1"
EVAL_PROTOCOL = "historical_full_dev_greedy_nf4_fp16"  # One greedy output, not 20-sample decoding.
EVAL_HELPER = PROJECT_ROOT / "cse_dpo/hea_historical_evaluation.py"

SEED = 3407
NUM_TRAIN_EPOCHS = 8
LEARNING_RATE = 5e-6
BATCH_QUESTIONS = 32
BETA = 0.1
LAMBDA_EQ = 0.1
LAMBDA_REAL = 0.1
MAX_SEQ_LENGTH = 9000  # Full evidence; overflow raises an error rather than truncating.
GENERATED_MAX_NEW_TOKENS = 64
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
EVALUATE_EACH_EPOCH = True

method_weights = {"standard_dpo": (0., 0.), "hea_dpo": (LAMBDA_EQ, LAMBDA_REAL),
                  "class_only": (0., 0.), "class_equivalence": (LAMBDA_EQ, 0.),
                  "class_realization": (0., LAMBDA_REAL)}
if METHOD not in method_weights:
    raise ValueError(f"Unknown METHOD: {METHOD}")
eq_weight, real_weight = method_weights[METHOD]
if RUN_NAME is None:
    RUN_NAME = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
if not isinstance(RUN_NAME, str) or not RUN_NAME.strip() or any(c in RUN_NAME for c in '/\\') or RUN_NAME in {'.', '..'}:
    raise ValueError("RUN_NAME must be a nonempty label without path separators.")
OUTPUT_DIR = PROJECT_ROOT / f"Artifacts/cse_dpo/hea_dpo_qwen25_{MODEL_SIZE}/runs" / f"{METHOD}_{DATASET_LABEL}_{MODEL_LABEL}_historical_eval_{RUN_NAME}_seed{SEED}{'_smoke' if SMOKE_TEST else ''}"
args = SimpleNamespace(data=TRAIN_JSONL, model=MODEL_REF, output=OUTPUT_DIR,
    epochs=1 if SMOKE_TEST else NUM_TRAIN_EPOCHS, learning_rate=LEARNING_RATE,
    batch_questions=2 if SMOKE_TEST else BATCH_QUESTIONS, beta=BETA,
    lambda_eq=eq_weight, lambda_real=real_weight, max_seq_length=MAX_SEQ_LENGTH, seed=SEED)
assert args.epochs > 0 and args.batch_questions > 0 and BETA > 0
assert 0 <= WARMUP_RATIO < 1 and min(eq_weight, real_weight) >= 0
for path in [TRAIN_JSONL, DEV_INPUT, GOLD_INPUT, EVAL_PROMPT_REGISTRY, EVAL_HELPER, SFT_PROMPT_INPUT, SFT_SELECTION_SUMMARY, MODEL_REF / 'adapter_config.json', BASE_MODEL_REF / 'config.json']:
    assert path.exists(), f"Missing input: {path}"
adapter_config = json.loads((MODEL_REF / "adapter_config.json").read_text())
recorded_base = Path(adapter_config["base_model_name_or_path"])
if recorded_base.name != BASE_MODEL_REF.name:
    raise ValueError(f"Adapter/base mismatch: {recorded_base} versus {BASE_MODEL_REF}")
print("Model size:", MODEL_SIZE, "| base:", BASE_MODEL_REF)
print("Method:", METHOD, "| HEA dataset:", HEA_DATASET_VARIANT, "| epochs:", args.epochs, "| effective question batch:", args.batch_questions)
print("SFT initialization and frozen reference:", MODEL_REF)
print("Output:", OUTPUT_DIR)


## 3. Objective and completion scoring

For each candidate, $r=\beta(\log\pi_\theta-\log\pi_{\rm SFT})$.

With `METHOD = "standard_dpo"`, each question visit first samples one of the reviewed class comparisons using its stored probability. It then samples one candidate uniformly from the preferred class and one from the rejected class and applies ordinary pairwise DPO:

$$\mathcal L_{\rm DPO}=\operatorname{softplus}(-(r_{\rm chosen}-r_{\rm rejected})).$$

This is a matched baseline: it uses the same questions, candidates, class-comparison distribution, SFT policy/reference, prompts, and number of question visits as HEA-DPO. It does not aggregate a class and has no equivalence or canonical-realization term.

For the HEA methods, a class score is $\log\operatorname{mean}\exp(r)$ over **all retained members**. The preference loss is $\operatorname{softplus}(-(S_i-S_j))$. Equivalence is the population variance of rewards over all C3 members, including when the sampled comparison excludes C3. Singleton C3 has zero variance. Realization is the token-averaged negative log-probability of the canonical answer.

Completions include the SFT space after `Answer:` and EOS; prompt tokens are masked out. `class_loss` in the saved history is retained for compatibility and equals the active preference loss; for `standard_dpo`, it is the ordinary pairwise DPO loss.


In [ ]:
def sha(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def objective(scores, refs, classes, preferred, rejected, canonical, lengths,
              beta=.1, lambda_eq=.05, lambda_real=.1, method='hea_dpo'):
    reward = beta * (scores - refs)
    if method == 'standard_dpo':
        if len(classes[preferred]) != 1 or len(classes[rejected]) != 1:
            raise ValueError('Standard DPO requires exactly one sampled member per selected class')
        margin = reward[classes[preferred][0]] - reward[classes[rejected][0]]
        preference_loss = F.softplus(-margin)
        zero = scores.sum() * 0.0  # differentiable zeros keep component logging uniform
        return preference_loss, (preference_loss, zero, zero, margin)

    def aggregate(indices):
        return torch.logsumexp(reward[indices], dim=0) - math.log(len(indices))
    margin = aggregate(classes[preferred]) - aggregate(classes[rejected])
    preference_loss = F.softplus(-margin)
    equiv = reward[classes['C3']].var(unbiased=False)
    realize = -scores[canonical] / lengths[canonical]
    return preference_loss + lambda_eq * equiv + lambda_real * realize, (preference_loss, equiv, realize, margin)

# CPU-safe formula checks. The standard branch must equal ordinary sequence-level DPO.
_test_scores = torch.tensor([-2.0, -3.0], requires_grad=True)
_test_refs = torch.tensor([-2.5, -2.75])
_test_classes = {'C3': [0], 'C1': [1]}
_test_lengths = torch.tensor([2.0, 2.0])
_test_loss, _ = objective(_test_scores, _test_refs, _test_classes, 'C3', 'C1', None,
                          _test_lengths, beta=0.1, lambda_eq=0.0, lambda_real=0.0,
                          method='standard_dpo')
_test_expected = F.softplus(-0.1 * ((_test_scores[0] - _test_refs[0]) -
                                    (_test_scores[1] - _test_refs[1])))
assert torch.allclose(_test_loss, _test_expected)
del _test_scores, _test_refs, _test_classes, _test_lengths, _test_loss, _test_expected

def encode(tokenizer, prompt, answer, max_length):
    prefix = tokenizer.encode(prompt, add_special_tokens=False)
    full = tokenizer.encode(prompt + ' ' + answer, add_special_tokens=False)
    if full[:len(prefix)] != prefix:
        raise ValueError('Prompt/completion tokenization boundary is not stable')
    full.append(tokenizer.eos_token_id)
    n = len(full) - len(prefix)
    if len(full) > max_length:
        raise ValueError(f'Sequence length {len(full)} exceeds {max_length}; truncation is forbidden')
    return full, n

def score(model, encoded, device):
    ids, n = encoded
    x = torch.tensor([ids], device=device)
    # Only materialize completion logits, including the prediction of EOS.
    logits = model(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False,
                   logits_to_keep=n + 1).logits[:, :-1, :].float()
    target = x[:, -n:]
    assert logits.shape[1] == n
    return -F.cross_entropy(logits.reshape(-1, logits.shape[-1]), target.reshape(-1), reduction='sum')

def prepare(args, tokenizer):
    rows = [json.loads(line) for line in args.data.read_text().splitlines() if line]
    manifest = json.loads((args.data.parent / 'manifest.json').read_text())
    assert sha(args.data) == manifest['output_hashes'][args.data.name]
    encoded, lengths = {}, []
    for q in rows:
        assert q['eligible_for_training'] and q['review_status'] == 'user_accepted'
        expected = tokenizer.apply_chat_template(q['messages'], tokenize=False, add_generation_prompt=True) + 'Answer:'
        assert expected == q['prompt'], 'Stored reviewed prompt/template mismatch'
        q['source_reviewed_prompt'] = q['prompt']
        q['messages'] = [dict(m) for m in q['messages']]
        assert q['messages'][0]['role'] == 'system'
        q['messages'][0]['content'] = SFT_INSTRUCTION
        q['prompt'] = tokenizer.apply_chat_template(q['messages'], tokenize=False, add_generation_prompt=True) + 'Answer:'
        members = [c for cs in q['classes'].values() for c in cs]
        for c in members:
            assert c['annotation']['review_status'] == 'reviewed' and c['annotation']['format_valid']
            pair = encode(tokenizer, q['prompt'], c['text'], args.max_seq_length)
            encoded[c['candidate_id']] = pair
            lengths.append(len(pair[0]))
        canonical = next(c for c in q['classes']['C3'] if c['canonical'])
        full_sft = tokenizer.apply_chat_template(q['messages'] + [{'role': 'assistant', 'content': 'Answer: ' + q['canonical_answer']}], tokenize=False, add_generation_prompt=False)
        expected_full = q['prompt'] + ' ' + canonical['text'] + tokenizer.eos_token + '\n'
        assert full_sft == expected_full, 'Canonical SFT completion mismatch'
        assert all(len(cs) <= 4 for cs in q['classes'].values())
        assert abs(sum(c['probability'] for c in q['comparisons']) - 1) < 1e-9
    report = {'status': 'passed', 'questions': len(rows), 'candidates': len(encoded),
              'max_sequence_tokens': max(lengths), 'prompt_and_canonical_sft_alignment': True,
              'truncation': False, 'eos_scored': True, 'data_sha256': sha(args.data),
              'prompt_policy': 'Selected strict-extractive SFT instruction; reviewed question/evidence/classes preserved',
              'sft_instruction_sha256': hashlib.sha256(SFT_INSTRUCTION.encode()).hexdigest()}
    return rows, encoded, report


## 4. Load and validate the reviewed data


In [ ]:
def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

def write_jsonl(path, rows):
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows))

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_REF), local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
sft_prepared = json.loads(SFT_PROMPT_INPUT.read_text())
sft_instructions = {clean_text(r['instruction']) for r in sft_prepared}
assert len(sft_instructions) == 1, "Expected one SFT instruction; select an explicit prompt if this changes."
SFT_INSTRUCTION = next(iter(sft_instructions))
sft_selection = json.loads(SFT_SELECTION_SUMMARY.read_text())
assert Path(sft_selection['model_dir']).resolve() == MODEL_REF.resolve()
assert sft_selection['aggregate']['by_type']['factoid']['primary_metric'] == 'mrr'
print('Selected SFT checkpoint: step', sft_selection['step'], '| recorded dev MRR:', sft_selection['metric_value'])
all_rows, all_encoded, preflight = prepare(args, tokenizer)
manifest = json.loads((TRAIN_JSONL.parent / "manifest.json").read_text())
for relative, expected_hash in manifest['source_hashes'].items():
    path = PROJECT_ROOT.parent / relative
    assert sha(path) == expected_hash, f"Reviewed source changed: {path}"
held = {json.loads(l)['question_id'] for l in (PILOT_ROOT / 'full_model_v1/question_review_queue.jsonl').read_text().splitlines()}
flagged = {json.loads(l)['candidate_id'] for l in (PILOT_ROOT / 'full_model_v1/review_queue.jsonl').read_text().splitlines()}
for q in all_rows:
    assert q['question_id'] not in held and q['split'] == 'train'
    assert all(c['candidate_id'] not in flagged for cs in q['classes'].values() for c in cs)
rows = all_rows[:SMOKE_TRAIN_QUESTIONS] if SMOKE_TEST else all_rows
active_ids = {c['candidate_id'] for q in rows for cs in q['classes'].values() for c in cs}
encoded = {cid: value for cid, value in all_encoded.items() if cid in active_ids}
assert rows

# Use the historical evaluator renderer, multiline prompt, and prepared gold.
dev_rows = prepare_historical_dev(tokenizer, DEV_INPUT, EVAL_PROMPT_REGISTRY, EVAL_PROMPT_ID,
    max_seq_length=MAX_SEQ_LENGTH, max_new_tokens=GENERATED_MAX_NEW_TOKENS,
    limit=SMOKE_DEV_QUESTIONS if SMOKE_TEST else None)
assert not {q['question_id'] for q in all_rows} & {q['question_id'] for q in dev_rows}
if not SMOKE_TEST:
    assert len(dev_rows) == 160
print('Evaluation protocol:', EVAL_PROTOCOL)
assert dev_rows
coverage = pd.DataFrame([{'subset': subset, 'questions': sum(q['pilot_subset'] == subset for q in rows),
    **{cls: sum(len(q['classes'][cls]) for q in rows if q['pilot_subset'] == subset) for cls in ['C3', 'C2', 'C1', 'C0']}}
    for subset in ['random', 'span_enriched']])
display(coverage)
print("Train:", len(rows), "questions /", len(encoded), "candidates; dev:", len(dev_rows))
print("Full training preflight:", preflight)
display(pd.DataFrame([{'class': cls, 'answer': c['text'], 'canonical': c['canonical']}
    for cls, members in rows[0]['classes'].items() for c in members]))


## 5. Create an isolated run and load the SFT policy
This cell requires CUDA. Re-run from a fresh kernel with a new run name to start another experiment.


In [ ]:
if DEVICE.type != 'cuda':
    raise RuntimeError("CUDA is required for training. Select your GPU-enabled bioasq kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This recipe requires a BF16-capable CUDA GPU.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
write_json(OUTPUT_DIR / 'preflight.json', preflight)
write_jsonl(OUTPUT_DIR / 'active_train_questions.jsonl', rows)
write_json(OUTPUT_DIR / 'active_dev_ids.json', [q['question_id'] for q in dev_rows])

def event(kind, **values):
    record = {'event': kind, 'time': time.time(), **values}
    with (OUTPUT_DIR / 'events.jsonl').open('a') as f:
        f.write(json.dumps(record) + '\n')
    if kind != 'question_trained':
        print(json.dumps(record))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
rng = random.Random(SEED)  # Shared order/comparison stream; identical across objectives.
candidate_rng = random.Random(SEED + 1_000_003)  # Standard-DPO member sampling only.
device = DEVICE
model = AutoModelForCausalLM.from_pretrained(str(BASE_MODEL_REF), dtype=torch.bfloat16,
    attn_implementation='sdpa', local_files_only=True)
model = PeftModel.from_pretrained(model, str(MODEL_REF), is_trainable=True, local_files_only=True).to(device)
model.config.use_cache = False
for module in model.modules():
    if isinstance(module, torch.nn.Dropout):
        module.p = 0.0  # Required for exact two-pass differentiation.
params = [p for p in model.parameters() if p.requires_grad]
assert params and all('lora_' in n for n, p in model.named_parameters() if p.requires_grad)
model.eval()
source_files = list(MODEL_REF.glob('*.safetensors')) + list(BASE_MODEL_REF.glob('*.safetensors'))
source_files += [EVAL_PROMPT_REGISTRY, EVAL_HELPER, SFT_PROMPT_INPUT, SFT_SELECTION_SUMMARY, MODEL_REF / 'adapter_config.json', BASE_MODEL_REF / 'config.json',
                 MODEL_REF / 'tokenizer.json', MODEL_REF / 'tokenizer_config.json', MODEL_REF / 'chat_template.jinja']
fingerprints = {str(p): sha(p) for p in source_files}
run_config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}
run_config.update(method=METHOD, hea_dataset_variant=HEA_DATASET_VARIANT, dataset_label=DATASET_LABEL, smoke_test=SMOKE_TEST, model_hashes=fingerprints,
    model_size=MODEL_SIZE, base_model_ref=str(BASE_MODEL_REF),
    model_label=MODEL_LABEL, sft_selection=sft_selection, sft_instruction=SFT_INSTRUCTION,
    active_train_sha256=sha(OUTPUT_DIR / 'active_train_questions.jsonl'),
    data_sha256=sha(TRAIN_JSONL), dev_sha256=sha(DEV_INPUT), gold_sha256=sha(GOLD_INPUT),
    active_train_questions=len(rows), active_dev_questions=len(dev_rows),
    warmup_ratio=WARMUP_RATIO, max_grad_norm=MAX_GRAD_NORM, generated_max_new_tokens=GENERATED_MAX_NEW_TOKENS,
    evaluate_each_epoch=EVALUATE_EACH_EPOCH, dropout=0.0, dtype='bfloat16', quantization=None,
    reference='Unchanged SFT scores computed before updates', torch_version=torch.__version__,
    gpu=torch.cuda.get_device_name(), final_checkpoint_selection='Fixed epoch budget',
    evaluation_protocol=EVAL_PROTOCOL, evaluation_prompt_id=EVAL_PROMPT_ID,
    evaluation_quantization='nf4', evaluation_compute_dtype='float16',
    evaluation_use_cache=False, evaluation_decoding='greedy',
    evaluation_gold_source=str(DEV_INPUT),
    objective_definition=('ordinary_sequence_pair_dpo' if METHOD == 'standard_dpo' else 'hea_class_logmeanexp'),
    pair_sampling=('one_uniform_candidate_per_selected_class' if METHOD == 'standard_dpo' else 'all_retained_members_of_selected_classes_and_C3'),
    order_and_comparison_seed=SEED, standard_dpo_candidate_seed=SEED + 1_000_003)
write_json(OUTPUT_DIR / 'config.json', run_config)
model.print_trainable_parameters()


## 6. Freeze reference scores
Score every retained candidate before updating the policy. The saved cache belongs to this run and is never recomputed using trained weights.


In [ ]:
reference = {}
with torch.no_grad():
    for cid, pair in tqdm(encoded.items(), desc='Frozen SFT reference'):
        reference[cid] = float(score(model, pair, device))
        assert math.isfinite(reference[cid])
write_json(OUTPUT_DIR / 'reference_scores.json', {'model_hashes': fingerprints,
    'data_sha256': sha(TRAIN_JSONL), 'scores': reference})
model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
with torch.no_grad():
    for cid in list(encoded)[:3]:
        assert abs(float(score(model, encoded[cid], device)) - reference[cid]) < 1e-3
print("Frozen reference candidates:", len(reference))


## 7. Historical development evaluation and checkpoint selection

Each evaluation loads the current adapter snapshot into a separate model with the historical settings: 4-bit NF4, double quantization, FP16 compute, one greedy completion, at most 64 new tokens, no generation cache, multiline strict-extraction instructions, and all development resources. The training policy and frozen-reference scores remain BF16.

The same repository generation function and scorer as the standalone evaluator are used. Gold answers come from `DEV_INPUT`, matching the historical run. Prompts, settings and per-question predictions are saved under `historical_evaluations/epoch_N/`. Context overflow raises an error.

Canonical NLL is measured on the quantized evaluation model and the historical prompt, so it is not directly comparable with the previous BF16 NLL. Format validity and exact match use the historical function's cleaned prediction. `prediction_tokens` counts the cleaned prediction's tokens, not the raw generated token count. The prior raw-generation format metric is not retained under the same definition.

The original evaluation notebook now offers 20-sample decoding, but the saved `20260908_120106` run used greedy decoding. This notebook explicitly reproduces that saved run's greedy protocol. It does not inherit the evaluator notebook's current sampling preset.

Save the initial SFT baseline and the best checkpoints by historical dev MRR and canonical NLL. Ties keep the earlier checkpoint. These are development-selected models, not test results. The fixed-budget final adapter is saved separately.

For the **0.5B adapter only**, alignment was verified on the full 160-question development set: the unchanged SFT adapter reproduced historical MRR **0.44375** and **160/160 identical predictions**. The saved comparison is [here](../Artifacts/Factoid_SFT/evaluations/hea_historical_baseline_check_20260909_033059_049844/historical_comparison.json). This is not a verified 3B baseline; the 3B run measures its own epoch-zero results.


In [ ]:
def valid_single_answer(text):
    match = re.fullmatch(r'\s*\[BE\]((?:(?!\[BE\]|\[EE\]).)+)\[EE\]\s*', text, re.DOTALL)
    return bool(match and match.group(1).strip())

def save_checkpoint(path):
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

eval_history = []
best_dev_nll = float('inf')
best_dev_mrr = -float('inf')
best_summaries = {}

def evaluate_dev(epoch, step):
    global best_dev_nll, best_dev_mrr
    # Evaluate saved weights through a separate historical NF4/FP16 model.
    # The BF16 training policy and optimizer remain intact.
    if epoch == 0:
        evaluation_adapter = MODEL_REF
    else:
        evaluation_adapter = OUTPUT_DIR / '_evaluation_adapters' / f'epoch_{epoch}'
        if evaluation_adapter.exists():
            raise FileExistsError(f'Evaluation snapshot already exists: {evaluation_adapter}')
        save_checkpoint(evaluation_adapter)
    summary, metrics = evaluate_historical_adapter(
        evaluation_adapter, BASE_MODEL_REF, dev_rows,
        OUTPUT_DIR / 'historical_evaluations' / f'epoch_{epoch}',
        max_seq_length=MAX_SEQ_LENGTH, max_new_tokens=GENERATED_MAX_NEW_TOKENS,
        progress_factory=tqdm, description=f'Historical dev epoch {epoch}')
    summary.update(epoch=epoch, step=step, smoke_test=SMOKE_TEST)
    eval_history.append(summary)
    write_jsonl(OUTPUT_DIR / f'dev_predictions_epoch_{epoch}.jsonl', metrics)
    write_json(OUTPUT_DIR / 'evaluation_history.json', eval_history)
    if summary['canonical_nll'] < best_dev_nll:
        best_dev_nll = summary['canonical_nll']
        best_summaries['canonical_nll'] = summary.copy()
        save_checkpoint(OUTPUT_DIR / 'best_model_by_dev_nll')
    if summary['mrr'] > best_dev_mrr:
        best_dev_mrr = summary['mrr']
        best_summaries['mrr'] = summary.copy()
        save_checkpoint(OUTPUT_DIR / 'best_model_by_dev_mrr')
    write_json(OUTPUT_DIR / 'best_dev_summaries.json', best_summaries)
    display(pd.DataFrame([summary]))
    return summary

initial_dev = evaluate_dev(epoch=0, step=0)


## 8. Train the selected preference objective

Each epoch visits every question once in shuffled order and samples a class comparison using its recorded within-question probabilities.

- `standard_dpo`: uniformly sample one member from each selected class and apply ordinary pairwise DPO. Candidate IDs are recorded for auditability.
- HEA methods: use every retained member of the selected classes and C3, then apply the configured class, equivalence, and realization terms.

To bound memory, candidate log-probabilities are first computed without activation graphs. The objective is differentiated with respect to these scalar scores, and each selected candidate is recomputed to backpropagate its coefficient. Dropout is disabled so the two passes agree. Logs distinguish sequence-score gradient norms from the total parameter-gradient norm.


In [ ]:
start_time = time.time()
optimizer = torch.optim.AdamW(params, lr=args.learning_rate, weight_decay=0)
total_steps = math.ceil(len(rows) / args.batch_questions) * args.epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=max(1, math.ceil(total_steps * WARMUP_RATIO)), num_training_steps=total_steps)
comparisons, sampled_candidate_pairs, exposure, history = Counter(), Counter(), Counter(), []
step = 0
event('training_start', epochs=args.epochs, optimizer_steps=total_steps, questions=len(rows))
for epoch in range(args.epochs):
    ordered = list(rows)
    rng.shuffle(ordered)
    for offset in range(0, len(ordered), args.batch_questions):
        batch = ordered[offset:offset + args.batch_questions]
        optimizer.zero_grad(set_to_none=True)
        batch_metrics = []
        for q in tqdm(batch, desc=f"Epoch {epoch + 1}, update {step + 1}", leave=False):
            comparison = rng.choices(q['comparisons'], weights=[c['probability'] for c in q['comparisons']])[0]
            hi, lo = comparison['preferred'], comparison['rejected']
            pair_event = {}
            if METHOD == 'standard_dpo':
                chosen = candidate_rng.choice(q['classes'][hi])
                rejected = candidate_rng.choice(q['classes'][lo])
                classes = {hi: [chosen], lo: [rejected]}
                members = [chosen, rejected]
                canonical = None
                pair_key = f"{chosen['candidate_id']}>{rejected['candidate_id']}"
                sampled_candidate_pairs[pair_key] += 1
                pair_event = {'chosen_candidate_id': chosen['candidate_id'],
                              'rejected_candidate_id': rejected['candidate_id']}
            else:
                classes = {cls: ms for cls, ms in q['classes'].items() if cls in {hi, lo, 'C3'}}
                members = [c for ms in classes.values() for c in ms]
            index = {c['candidate_id']: i for i, c in enumerate(members)}
            class_indices = {cls: [index[c['candidate_id']] for c in ms] for cls, ms in classes.items()}
            if METHOD != 'standard_dpo':
                canonical = next(index[c['candidate_id']] for c in classes['C3'] if c['canonical'])
            with torch.no_grad():
                scores = torch.stack([score(model, encoded[c['candidate_id']], device) for c in members])
            scores.requires_grad_(True)
            refs = torch.tensor([reference[c['candidate_id']] for c in members], device=device)
            lengths = torch.tensor([encoded[c['candidate_id']][1] for c in members], device=device)
            loss, components = objective(scores, refs, class_indices, hi, lo, canonical, lengths,
                                         args.beta, args.lambda_eq, args.lambda_real, method=METHOD)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite loss')
            # Log component gradients in sequence-score space, labeled explicitly.
            component_norms = [float(torch.autograd.grad(v, scores, retain_graph=True)[0].norm()) for v in components[:3]]
            coefficients = torch.autograd.grad(loss, scores)[0].detach() / len(batch)
            for i, c in enumerate(members):
                current = score(model, encoded[c['candidate_id']], device)
                if not torch.allclose(current.detach(), scores[i].detach(), atol=1e-3, rtol=1e-5):
                    raise RuntimeError('Two-pass score mismatch')
                (current * coefficients[i]).backward()
            comparisons[f'{hi}>{lo}'] += 1
            exposure[q['question_id']] += 1
            metrics = dict(zip(['preference_loss', 'equivalence_loss', 'realization_loss', 'margin'], [float(v.detach()) for v in components]))
            metrics['class_loss'] = metrics['preference_loss']  # backward-compatible history column
            metrics.update(loss=float(loss.detach()), score_gradient_norm_class=component_norms[0],
                           score_gradient_norm_equivalence=component_norms[1], score_gradient_norm_realization=component_norms[2])
            batch_metrics.append(metrics)
            event('question_trained', epoch=epoch + 1, question_id=q['question_id'],
                  comparison=f'{hi}>{lo}', **pair_event, **metrics)
        grad_norm = torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM, error_if_nonfinite=True)
        lr_used = optimizer.param_groups[0]['lr']
        optimizer.step()
        scheduler.step()
        step += 1
        metric = {k: sum(m[k] for m in batch_metrics) / len(batch_metrics) for k in batch_metrics[0]}
        metric.update(step=step, epoch=epoch + 1, question_groups=len(batch), gradient_norm=float(grad_norm), learning_rate=lr_used)
        history.append(metric)
        event('optimizer_step', **metric)
    checkpoint = args.output / f'epoch_{epoch + 1}'
    model.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    event('epoch_saved', epoch=epoch + 1, path=str(checkpoint))

    if EVALUATE_EACH_EPOCH or epoch == args.epochs - 1:
        evaluate_dev(epoch=epoch + 1, step=step)

elapsed = time.time() - start_time
print(f"Training and epoch evaluation finished in {elapsed / 60:.2f} minutes")


## 9. Save final adapter, histories, and plots


In [ ]:
save_checkpoint(OUTPUT_DIR / 'final_adapter')
assert all(sha(Path(p)) == value for p, value in fingerprints.items()), 'SFT/base source files changed'
assert len(exposure) == len(rows) and all(n == args.epochs for n in exposure.values())
history_df = pd.DataFrame(history)
eval_df = pd.DataFrame(eval_history)
history_df.to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
eval_df.to_csv(OUTPUT_DIR / 'evaluation_history.csv', index=False)
summary = {'status': 'completed', 'method': METHOD, 'smoke_test': SMOKE_TEST,
    'model_size': MODEL_SIZE, 'base_model_ref': str(BASE_MODEL_REF),
    'questions': len(rows), 'candidates': len(encoded), 'epochs': args.epochs,
    'optimizer_steps': step, 'elapsed_seconds_training_and_epoch_eval': elapsed,
    'comparison_counts': dict(comparisons),
    'sampled_candidate_pair_counts': dict(sampled_candidate_pairs) if METHOD == 'standard_dpo' else None,
    'question_exposure': dict(exposure),
    'initial_dev': initial_dev, 'final_dev': eval_history[-1], 'best_dev': best_summaries,
    'reference_unchanged': True, 'final_adapter': str(OUTPUT_DIR / 'final_adapter'),
    'peak_cuda_memory_allocated_bytes': torch.cuda.max_memory_allocated()}
write_json(OUTPUT_DIR / 'summary.json', summary)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for key in ['loss', 'preference_loss', 'realization_loss']:
    axes[0, 0].plot(history_df['step'], history_df[key], label=key)
axes[0, 0].set_title('Training losses (raw components)')
axes[0, 1].plot(history_df['step'], history_df['equivalence_loss'], label='equivalence_loss')
axes[0, 1].set_title('C3 reward variance')
for key in ['mrr', 'top1', 'single_answer_format_valid', 'canonical_exact_match']:
    axes[1, 0].plot(eval_df['step'], eval_df[key], marker='o', label=key)
axes[1, 0].set_title('Development generation metrics')
axes[1, 0].set_ylim(-.03, 1.03)
axes[1, 1].plot(eval_df['step'], eval_df['canonical_nll'], marker='o', label='dev canonical NLL')
axes[1, 1].set_title('Development canonical NLL')
for ax in axes.flat:
    ax.set_xlabel('Optimizer step')
    ax.grid(alpha=.2)
    ax.legend()
fig.suptitle(f"{METHOD} — {'SMOKE SUBSET' if SMOKE_TEST else 'full reviewed training / full dev'}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()
display(eval_df)
print('Final adapter:', OUTPUT_DIR / 'final_adapter')


## 10. Reload and verify the saved adapter
This checks checkpoint integrity and scoring consistency. The development metrics above determine answer quality; successful loading alone does not.


In [ ]:
from safetensors.torch import load_file
original_weights = load_file(str(MODEL_REF / 'adapter_model.safetensors'))
saved_weights = load_file(str(OUTPUT_DIR / 'final_adapter/adapter_model.safetensors'))
assert original_weights.keys() == saved_weights.keys()
assert all(bool(torch.isfinite(v).all()) for v in saved_weights.values())
changed = sum(not torch.equal(original_weights[k], saved_weights[k]) for k in original_weights)
assert changed > 0, 'No adapter parameters changed; inspect learning rate and update count.'
check_q = rows[0]
check_c = next(c for c in check_q['classes']['C3'] if c['canonical'])
model.eval()
with torch.no_grad():
    before_reload = float(score(model, encoded[check_c['candidate_id']], device))
del original_weights, saved_weights, optimizer, scheduler, params, model
gc.collect()
torch.cuda.empty_cache()
reloaded_base = AutoModelForCausalLM.from_pretrained(str(BASE_MODEL_REF), dtype=torch.bfloat16,
    attn_implementation='sdpa', local_files_only=True)
model = PeftModel.from_pretrained(reloaded_base, str(OUTPUT_DIR / 'final_adapter'), local_files_only=True).to(device).eval()
with torch.no_grad():
    after_reload = float(score(model, encoded[check_c['candidate_id']], device))
assert math.isclose(before_reload, after_reload, abs_tol=1e-3, rel_tol=1e-5)
verification = {'status': 'passed', 'changed_adapter_tensors': changed,
    'finite_weights': True, 'reload_score_matches': True,
    'adapter_sha256': sha(OUTPUT_DIR / 'final_adapter/adapter_model.safetensors')}
write_json(OUTPUT_DIR / 'verification.json', verification)
display(verification)


## Outputs and interpretation

The run directory contains `final_adapter/`, each `epoch_N/`, `best_model_by_dev_mrr/`, `best_model_by_dev_nll/`, reference scores, per-question logs, development predictions, histories, plots, configuration/hashes, and checkpoint verification.

Use `final_adapter/` for the fixed-budget comparison with the earlier run. A development-selected checkpoint may be the unchanged SFT baseline; consult `best_dev_summaries.json`. Test questions have not been loaded by this notebook. Compare the chosen model on a separate test set after fixing the training and selection protocol. MRR can credit one answer within multiple generated spans, so inspect single-answer validity and `format_valid_top1` alongside MRR.

To run the matched baseline, restart the kernel, set `METHOD = "standard_dpo"`, choose a new run name, and run every cell. To repeat with a new seed or objective ablation, restart the kernel, change the configuration, and use a new run name. Full runs can take longer than the CLI training-only run because they also generate development predictions before training and after each epoch.
